# RecOps на Kaggle: обучить всё одним прогоном

**Перед запуском (один раз):**
1. Загрузите архив `recops-kaggle.zip` как Kaggle Dataset (Create → New Dataset), назовите его `recops-code`, затем в этом блокноте нажмите *Add Input* и выберите его.
2. Справа в *Session options*: **Internet = On** (нужен для скачивания Yambda с Hugging Face и `pip`), **Accelerator = None** (GPU не нужен), **Persistence = Files only**.
3. Выберите профиль в ячейке ниже. Порядок: `synthetic` → `smoke` → `full`. Пока не прошёл `synthetic`, на `full` не переходите.

Профили: `synthetic` (сгенерированные данные, без скачивания, проверяет код), `smoke` (реальные данные, маленький срез), `full` (Yambda-50M целиком).
Если ячейка упала, скопируйте последние ~40 строк вывода и отправьте мне.

In [ ]:
PROFILE = "synthetic"  # "synthetic" | "smoke" | "full"
WORKDIR = "/kaggle/working/recops_run"

In [ ]:
import glob
import os
import pathlib
import subprocess
import sys

CODE = None
for pp in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True):
    if (pathlib.Path(pp).parent / "src" / "recops").exists():
        CODE = pathlib.Path(pp).parent
        break
assert CODE is not None, "Не нашёл код: добавьте Dataset `recops-code` через Add Input"
print("code:", CODE)

ENV = dict(os.environ, PYTHONPATH=f"{CODE}/src", RECOPS_CONFIG_DIR=f"{CODE}/configs")
os.environ.update(PYTHONPATH=ENV["PYTHONPATH"], RECOPS_CONFIG_DIR=ENV["RECOPS_CONFIG_DIR"])


def sh(cmd):
    p = subprocess.Popen(
        cmd, shell=True, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    assert rc == 0, f"команда упала (код {rc}): {cmd}"


def step(name, extra=""):
    sh(f"{sys.executable} -m recops.cli {name} --profile {PROFILE} --workdir {WORKDIR} {extra}")

In [ ]:
# Зависимости (pip пропустит то, что уже установлено)
sh(
    f'{sys.executable} -m pip install -q "polars>=1.0" scipy pydantic pyyaml implicit catboost mlflow huggingface_hub fastapi httpx pytest'
)
import catboost
import implicit
import mlflow
import polars

print(
    "polars",
    polars.__version__,
    "| implicit",
    implicit.__version__,
    "| catboost",
    catboost.__version__,
    "| mlflow",
    mlflow.__version__,
)

## 0. Юнит-тесты (быстро; проверяют метрики, PSI, правило промоута, утечки во времени, API)

In [ ]:
sh(f'cd {CODE} && {sys.executable} -m pytest -q -p no:cacheprovider -m "not slow" tests')

## 1. Данные: скачивание, подготовка, валидация

In [ ]:
step("download")

In [ ]:
step("prepare")

In [ ]:
step("validate")

## 2. Скользящая симуляция продакшена
Каждый цикл: обучение кандидата → оценка на отложенной неделе → сравнение с production-моделью из MLflow Registry → решение (промоут или отклонение) → мониторинг PSI и качества.
По умолчанию цикл 2 содержит искусственный дрейф, а последний цикл обучает заведомо плохую модель, которую гейт должен отклонить.

In [ ]:
step("simulate")

## 3. Финальная оценка на тесте (запускается один раз; повторный запуск заблокирован)

In [ ]:
step("final")

## 4. Замер задержки API и итоговый отчёт

In [ ]:
step("bench")

In [ ]:
step("report")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open(f"{WORKDIR}/artifacts/report.md", encoding="utf-8").read()))

In [ ]:
import zipfile

art = pathlib.Path(WORKDIR) / "artifacts"
out = "/kaggle/working/recops_results_light.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["report.md", "*.json", "cycles/*.json", "final/results.json"]:
        for f in art.glob(pat):
            z.write(f, f.relative_to(art))
    z.write(pathlib.Path(WORKDIR) / "data/processed/meta.json", "meta.json")
print(out, round(os.path.getsize(out) / 1e6, 2), "MB")

## 5. Что сохранить
Всё лежит в `/kaggle/working/recops_run/`: `artifacts/report.md` (итоговый отчёт с цифрами из этого прогона), `artifacts/simulation.json`, `artifacts/final/results.json`, `artifacts/mlflow/` (SQLite-база MLflow, реестр и артефакты), `artifacts/final/bundle` (модель).
Чтобы результаты остались после закрытия сессии: **Save Version → Save & Run All (Commit)**, затем скачайте Output.